In [1]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer


c:\Users\skadi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
policy_text='''Food Delivery Refund Policy

Customers may request a refund if the delivered order is incorrect, damaged, or missing items.
Refund requests should normally be submitted within 24 hours after delivery.
If the restaurant cancels the order before preparation begins, the customer receives a full refund.
If the delivery partner cannot reach the customer because of an incorrect address, refunds may not be issued.
Orders delayed because of weather or heavy traffic may not qualify for a refund unless the food arrives damaged.
Partial refunds may be provided for missing items.
Refunds are usually processed within 5 to 7 business days.
Digital wallet refunds may appear faster than bank refunds.
Evidence such as photos may be requested for damaged food complaints.
Repeated abuse of the refund policy may result in account restrictions.'''
with open("refund_policy.txt","w",encoding="utf-8") as f:f.write(policy_text)


In [3]:
text=open("refund_policy.txt",encoding="utf-8").read()
def chunk_text(text,chunk_size=200):
    return [text[i:i+chunk_size] for i in range(0,len(text),chunk_size)]
chunks=chunk_text(text)
model=SentenceTransformer("all-MiniLM-L6-v2")
emb=np.array(model.encode(chunks)).astype("float32")
index=faiss.IndexFlatL2(emb.shape[1]); index.add(emb)
def retrieve(q,k=2):
    qv=np.array(model.encode([q])).astype("float32")
    _,idx=index.search(qv,k)
    return [chunks[i] for i in idx[0]]
def build_prompt(q,ctx):
    return f"""You are a customer support assistant.

Context:
{'\n\n'.join(ctx)}

Question:
{q}

Answer:
"""
while True:
    q=input("Ask a question (or quit): ")
    if q.lower()=="quit":
        print("Goodbye!"); break
    ctx=retrieve(q)
    print("\nRetrieved Chunks:")
    for i,c in enumerate(ctx,1):
        print(f"\nChunk {i}:\n{c}")
    print("\nPrompt:\n",build_prompt(q,ctx))
    print("\nPlaceholder Answer:")
    print("Simulated Answer: Based on the retrieved refund policy, follow the refund rules above.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5600.67it/s]



Retrieved Chunks:

Chunk 1:
be issued.
Orders delayed because of weather or heavy traffic may not qualify for a refund unless the food arrives damaged.
Partial refunds may be provided for missing items.
Refunds are usually proce

Chunk 2:
Food Delivery Refund Policy

Customers may request a refund if the delivered order is incorrect, damaged, or missing items.
Refund requests should normally be submitted within 24 hours after delivery.

Prompt:
 You are a customer support assistant.

Context:
be issued.
Orders delayed because of weather or heavy traffic may not qualify for a refund unless the food arrives damaged.
Partial refunds may be provided for missing items.
Refunds are usually proce

Food Delivery Refund Policy

Customers may request a refund if the delivered order is incorrect, damaged, or missing items.
Refund requests should normally be submitted within 24 hours after delivery.

Question:
Can I get a refund if my order is damaged?

Answer:


Placeholder Answer:
Simulated Ans